# Basira — train a Kraken Arabic **handwriting** model (Muharaf)

### How to run
1. **Runtime → Change runtime type → GPU (T4).**
2. **Runtime → Run all.**

That's it. Re-running is safe — the install and data-prep steps **skip themselves
if already done**, so you never have to worry about which cells to run. Cell 4
(training) is the long one (~1–3 h on a T4).

⚠️ **Licence:** Muharaf is **CC BY-NC-SA (non-commercial)** — R&D / demo use only.
Verified against **Kraken 7.x**.


In [ ]:
# 1) Install Kraken (safe to re-run)
!pip install -q kraken datasets pillow
import torch
print("CUDA available:", torch.cuda.is_available())   # must print True
# If pip prints a "restart" notice: Runtime > Restart session, then Run all again.


In [ ]:
# 2) Build Kraken training pairs — SKIPS automatically if already prepared
import os, glob
from datasets import load_dataset

if len(glob.glob("data/train/*.png")) > 100:
    print("data/ already prepared — skipping (re-run-safe)")
else:
    ds = load_dataset("aamijar/muharaf-public")        # public; no token needed
    # QUICK TEST? Uncomment for a small subset first (~20 min):
    # ds["train"] = ds["train"].select(range(2000))
    for split in ds:                                    # train / validation / test
        out = f"data/{split}"; os.makedirs(out, exist_ok=True); n = 0
        for i, row in enumerate(ds[split]):
            text = (row.get("text") or "").strip()
            if not text:
                continue
            img = row["image"]
            if img.mode not in ("RGB", "L"):
                img = img.convert("RGB")
            img.save(f"{out}/{i:06d}.png")
            with open(f"{out}/{i:06d}.gt.txt", "w", encoding="utf-8") as fh:
                fh.write(text)
            n += 1
        print(split, "->", n, "line pairs")


In [ ]:
# 3) Fetch the OpenITI Arabic base model (safe to re-run; cached after first time)
!kraken get 10.5281/zenodo.7050270
import glob, os
base = glob.glob(os.path.expanduser("~/.local/share/htrmopo/**/*.mlmodel"), recursive=True)[0]
print("base model:", base)


In [ ]:
# 4) TRAIN (the long step). Compiles a binary dataset, then transfer-learns
# from the Arabic base model. --force-type baseline matches the line data to the
# baseline base model; -o is a directory; output is .safetensors.
import glob, os
base = glob.glob(os.path.expanduser("~/.local/share/htrmopo/**/*.mlmodel"), recursive=True)[0]
if not glob.glob("muharaf_train.arrow"):
    !ketos compile -f path --force-type baseline -o muharaf_train.arrow data/train/*.png
cmd = f'ketos -d cuda:0 train -f binary --resize union -i "{base}" -o muharaf_hw muharaf_train.arrow'
print(cmd)
!{cmd}
# Watch val_accuracy each epoch -> held-out character accuracy (CER ~= 1 - val_accuracy).
# Training auto-stops when it plateaus; best model saved as muharaf_hw/best_*.safetensors


In [ ]:
# 5) Locate the best trained model
import glob
best = sorted(glob.glob("muharaf_hw/*.safetensors"))
print("best model:", best[-1] if best else "NONE - check the training output above")


In [ ]:
# 6) Save the model to Google Drive (then download it from drive.google.com)
import glob, shutil
best = sorted(glob.glob("muharaf_hw/*.safetensors"))[-1]
from google.colab import drive
drive.mount("/content/drive")
shutil.copy(best, "/content/drive/MyDrive/muharaf_hw_best.safetensors")
print("Saved to Drive as muharaf_hw_best.safetensors  (from", best + ")")


## Back on your Mac
Download `muharaf_hw_best.safetensors` from Google Drive, then with the app +
Kraken sidecar running:
```bash
scripts/install-kraken-model.sh ~/Downloads/muharaf_hw_best.safetensors muharaf
```
Set `DEMO_TRANSCRIBE_ADAPTER=kraken-muharaf` in `.env`, restart (`pnpm dev`),
log in as `demo@basira.test`, and transcribe — it runs your handwriting model.
